# Adaptive LoRA Router

Notebook 01

Environment Setup

Goal:

• Verify Colab GPU
• Install dependencies
• Load Qwen 3B
• Verify inference
• Measure GPU memory

Expected Output:

A working Qwen model ready for LoRA fine tuning.

### Step 1
Verify colab T4 gpu

In [ ]:
import torch
print("Is CUDA available?", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Installing unsloth dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

### Step 2
Import unlosth and download qwen-2.5 3B model

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit",
    load_in_4bit=True,
)


### Inference with the base Qwen model

In [ ]:
FastModel.for_inference(model)
messages = [
    {"role": "user", "content": "Explain what LoRA is in simple terms."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)
print(outputs)


In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### GPU memory cell

In [ ]:
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print(f"GPU Memory : {gpu_memory:.2f} GB")
print(f"Allocated : {allocated:.2f} GB")
print(f"Reserved : {reserved:.2f} GB")

## Trainable parameters

In [ ]:
def print_trainable_parameters(model):

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print(f"Total Parameters     : {total:,}")
    print(f"Trainable Parameters : {trainable:,}")